# Backtrader for Backtesting(Python)


## What is Backtrader 
Backtrader is a Python library that aids in strategy development and testing for traders of the financial markets.

It is an open-source framework that allows for strategy testing on historical data. Further, it can be used to optimize strategies, create visual plots, and can even be used for live trading.

there are other libraries also for backtesting like Zipline 

Here are some of the things Backtrader excels at:

### Backtesting – 
This might seem like an obvious one but Backtrader removes the tedious process of cleaning up your data and iterating through it to test strategies. It has built-in templates to use for various data sources to make importing data easier.

### Optimizing –
Adjusting a few parameters can sometimes be the difference between a profitable strategy and an unprofitable one. After running a backtest, optimizing is easily done by changing a few lines of code.

### Plotting – 
If you’ve worked with a few Python plotting libraries, you’ll know these are not always easy to configure, especially the first time around. A complex chart can be created with a single line of code.

### Indicators – 
Most of the popular indicators are already programmed in the Backtrader platform. This is especially useful if you want to test out an indicator but you’re not sure how effective it will be. Rather than trying to figure out the math behind the indicator, and how to code it, you can test it out first in Backtrader, probably with one line of code.

### Support for Complex Strategies –
Want to take a signal from one dataset and execute a trade on another? Does your strategy involve multiple timeframes? Or do you need to resample data? Backtrader has accounted for the various ways traders approach the markets and has extensive support.

$Note $:- 

        Backtrader is used for backtesting and not live trading 
        QuantConnect is a browser based platform that allows both backtesting and live trading 

## How Backtrader Works 
Backtrader shows you how your strategy might perform in the market by testing it against past price data.

The library’s most basic functionality is to iterate through historical data and to simulate the execution of trades based on signals given by your strategy.

It extends on this functionality in many ways. A Backtrader “analyzer” can be added to provide useful statistics. We will show an example of this using the commonly used Sharpe Ratio in a optimization test later in this tutorial.

On the subject of optimization, it’s clear a lot of thought has been put in to speeding up the testing of strategies with different parameters. The built in optimization module uses multiprocessing, fully utilizing your multiple CPU cores to speed up the process.

Lastly, Backtrader utilizes the well-known matplotlib library to create charts at the end of your backtest, if desired.

## How to configure the basic Bactrader Setup 

In [2]:
import backtrader as bt

In [3]:
class MyStrategy(bt.Strategy):
    def next(self):
        pass # Do Something 

# initiate Cerebro engine 
cerebro = bt.Cerebro()

# Add Strategy to Cerebro 
cerebro.addstrategy(MyStrategy)

# Run the Cerebro Engine
cerebro.run()

[]

We will go into the strategy class in more detail in the examples that follow. This is where all the logic goes in determining and executing your trade signals. It is also where indicators can be created or called, and where you can determine what get’s logged or printed to screen.

The cerebro engine is the core of Backtrader. This is the main class and we will add our data and strategies to it before eventually calling the cerebro.run() command.

## Adding the data and import it to Backtrader 

I will be using yfinance library to get the data 

In [4]:
import yfinance as yf 
import datetime as dt 

In [5]:
start = dt.datetime(2023,1,1)
end = dt.datetime(2026,1,1)
data = yf.download('TSLA',start,end)
data.columns = data.columns.droplevel(1)

[*********************100%***********************]  1 of 1 completed


In [6]:
data.to_csv('TSLA.csv')

In [7]:
data = bt.feeds.YahooFinanceCSV(dataname='TSLA.csv')
# cerebro.addata(data) for adding data to cerebro 

In the above example, we’ve assigned the CSV dataset to a variable named data. The next step is to add this to cerebro.

Adding data can be done at any point between instantiating cerebro and calling the cerebro.run() command. There are several additional parameters we can specify when loading our data. We will explore this further in our next example

» Here are some (mostly) free data sources and guides:

* Quandl: A Step-by-Step Guide
* Google Finance API and 9 Alternatives
* Yahoo Finance API – A Complete Guide

## How to print or log data using the strategy class in Backtrader 

### In the __init__ function above,
 we’ve created a variable called dataclose to make it easier to refer to the closing price later on. You will notice that the closing price is stored in datas[0].close. We can just as easily access the open price by referencing datas[0].open

An important feature of Backtrader is accessing historical data which we can now do via the dataclose variable. As Backtrader iterates through historical data, this variable will get updated with the latest price from dataclose[0]. We can also look back to the prior data points by accessing the negative index of dataclose. Here is an example.

if dataclose[0] > dataclose [-1]: pass # do something

The above code checks to see if the most recent close is larger than the prior close. We can just as easily access the second last closing price by changing the index like this: dataclose[-2]

In [8]:
class PrintClose(bt.Strategy):
    """The log function allows us to pass in data via the txt variable that we want to output to the screen. It will attempt to grab datetime values from the most recent data point,if available, and log it to the screen"""
    def log(self, txt , dt=None):
        dt = dt or self.datas[0].datetime.date(0)
        print(dt.isoformat(), txt) # print date and close
    def __init__(self):
        self.dataclose = self.datas[0].close
    def next(self):
        self.log(f'Close:{self.dataclose[0]}')

cerebro = bt.Cerebro()
# there is an issue with this since modern csvs data don't have adj Close columns but YahooFinanceCsvdata function assumes it still exists therefore its returning Volume instead of Closing price 
# data = bt.feeds.YahooFinanceCSVData(dataname='TSLA.csv')
data = bt.feeds.GenericCSVData(
    dataname='TSLA.csv',
    
    dtformat='%Y-%m-%d', # Ensure this matches the date format in your CSV (e.g., YYYY-MM-DD)
    
    datetime=0,
    open=4,
    high=2,
    low=3,
    close=1,
    volume=5,
    
    openinterest=-1, # -1 tells Backtrader this column doesn't exist in your CSV
    headers=True     # Assumes the first row of your CSV is the header (Date, Open, etc.)
)
cerebro.adddata(data)

cerebro.addstrategy(PrintClose)

cerebro.run()

        

2023-01-03 Close:108.0999984741211
2023-01-04 Close:113.63999938964844
2023-01-05 Close:110.33999633789062
2023-01-06 Close:113.05999755859375
2023-01-09 Close:119.7699966430664
2023-01-10 Close:118.8499984741211
2023-01-11 Close:123.22000122070312
2023-01-12 Close:123.55999755859375
2023-01-13 Close:122.4000015258789
2023-01-17 Close:131.49000549316406
2023-01-18 Close:128.77999877929688
2023-01-19 Close:127.16999816894531
2023-01-20 Close:133.4199981689453
2023-01-23 Close:143.75
2023-01-24 Close:143.88999938964844
2023-01-25 Close:144.42999267578125
2023-01-26 Close:160.27000427246094
2023-01-27 Close:177.89999389648438
2023-01-30 Close:166.66000366210938
2023-01-31 Close:173.22000122070312
2023-02-01 Close:181.41000366210938
2023-02-02 Close:188.27000427246094
2023-02-03 Close:189.97999572753906
2023-02-06 Close:194.75999450683594
2023-02-07 Close:196.80999755859375
2023-02-08 Close:201.2899932861328
2023-02-09 Close:207.32000732421875
2023-02-10 Close:196.88999938964844
2023-02-13

From this point on, the structure of our script will mostly remain the same and we will write the bulk of our strategies under the next function of the Strategy class.

## How to run a bactest using Backtrader 

We will test out a moving average crossover strategy. Essentially, it involves monitoring two moving averages and taking a trade when one crosses the other.

The moving average crossover strategy is to trading what the Hello World script is to programming. Neither will likely ever be used in the real world and are mostly used for illustrative purposes. In other words, we don’t expect the strategy to be a profitable one.

There are a few things we will do before diving into the strategy. First, we will separate our strategy into its own file. Throughout this tutorial, we will go over several examples and separating out the strategies from the main script will keep the code in a nice clean format.

The main script, which will have everything cerebro related, will only have minor changes throughout the tutorial while most of the work will be done in the strategies script.

We also have to separate our data into two parts. This way, we can test our strategy on the first part, run some optimization, and then see how it performs with our optimized parameters on the second set of data.

If you’ve heard the terms in-sample data, or out-of-sample data, this is what it is referring to. Out-of-sample data is simply data set aside for testing after optimization.

There are a lot of benefits to testing and optimizing this way, take a look at What is a Walk-Forward Optimization and How to Run It? if you’d like to get a more thorough understanding of the methodology.

To divide the data, we set a from date and to date when loading our data. Don’t forget to import the DateTime module for this part.

## "Refer to main.py and strategy2.py file in the github repo for bactrader " 

## How to build a stock screener in Backtrader 

Screeners are commonly used to filter out stocks based on certain parameters. There aren’t a lot of easy ways to look back to a certain date and see what results a stock screener might have spit out. Fortunately, Backtrader offers exactly this


We will test out this functionality by building a screener that filters out stocks that are trading two standad devation below the average price over the prior 20 days 

In [9]:
import backtrader as bt 
class Screener_SMA(bt.Analyzer):
    params = (('period',20),
              ('devfactor',2),)
    def start(self):
        self.bband = {
            data : bt.indicators.BollingerBands(data,period = self.params.period , devfactor = self.params.devfactor)
            for data in self.datas
        } # dictionary made for data and bband 
    
    def stop(self):
        self.rets['over'] = list()
        self.rets['under'] = list()

        for data ,band in self.bband.items():
            node = data._name , data.close[0] , round(band.lines.bot[0],2)
            # band.lines.bot[0] gives the MA-2SD for current data 
            if data > band.lines.bot :
                self.rets['over'].append(node)
            else:
                self.rets['under'].append(node)


Under the start function, you’ll notice that we are using Bollinger bands to determine the value for two standard deviations. The syntax is a bit different from prior examples as several datasets are used in a screener.

The stop function is where a bulk of our code falls. We iterate through our Bollinger band items for all of our datasets to filter out the ones that are trading below the lower band.

The stocks that qualify then get appended to a list. The analyzer class has a built-in dictionary with the variable name rets. We will use this dictionary to store our lists.

There isn’t a lot of code required in our main script, but it is quite different from prior examples. Since we are adding several datasets, we’ve created a list of all the tickers that we want to scan. We then iterate through the list to add the corresponding CSV files to cerebro.

The writer=True parameter calls the built-in writer functionality to display the ouput. stdstats=False removes some of the standard output (more on this later). And lastly, runonce=False ensures that data remains synchronized.

In [10]:
import datetime
import backtrader as bt

#Instantiate Cerebro engine
cerebro = bt.Cerebro()

#Add data to Cerebro
instruments = ['TSLA', 'AAPL', 'GE', 'GRPN','RELIANCE.NS','INFY']
for ticker in instruments:
    start = datetime.datetime(2025,1,1)
    end = datetime.datetime(2026,7,8)
    df = yf.download(ticker,start,end)
    df.columns = df.columns.droplevel(1)
    df.to_csv(f'{ticker}.csv')
    data = bt.feeds.GenericCSVData(
        dataname = f'{ticker}.csv',
        dtformat='%Y-%m-%d', # Ensure this matches the date format in your CSV (e.g., YYYY-MM-DD)
        datetime=0,
        open=4,
        high=2,
        low=3,
        close=1,
        volume=5,
        openinterest=-1,
        headers=True)
    cerebro.adddata(data) 

#Add analyzer for screener
cerebro.addanalyzer(Screener_SMA)

if __name__ == '__main__':
    #Run Cerebro Engine
    cerebro.run(runonce=False, stdstats=False, writer=True)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Cerebro:
  -----------------------------------------------------------------------------
  - Datas:
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data0:
      - Name: TSLA
      - Timeframe: Days
      - Compression: 1
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data1:
      - Name: AAPL
      - Timeframe: Days
      - Compression: 1
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data2:
      - Name: GE
      - Timeframe: Days
      - Compression: 1
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data3:
      - Name: GRPN
      - Timeframe: Days
      - Compression: 1
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data4:
      - Name: RELIANCE.NS
      - Timeframe: Days
      - Compression: 1
    +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    - Data5:
      -

## How to code an indicator in backtrader 

There are three ways to use an indicator in backtrader.

* you can use inbuilt indicator
 
* use a third party library 

* you can code from scratch 


Here We are gonna code ATR (Volatility Indicator) for our case 

In [16]:
class AverageTrueRange(bt.Strategy):
    def log(self,txt,dt=None):
        dt = dt or self.datas[0].datetime.date(0)
        print(f'{dt.isoformat()} {txt}') # print data and close 

    def __init__(self):
        self.datclose = self.datas[0].close
        self.datahigh = self.datas[0].high
        self.datalow = self.datas[0].low
    
    def next(self):
        range_total = 0
        
        for i in range(-13,1):
            true_range = self.datahigh[i] - self.datalow[i]

            range_total += true_range
        
        ATR = range_total / 14 

        self.log(f'Close: {self.dataclose[0]:.2f},ATR: {ATR:.4f}')


cerebro = bt.Cerebro()
data = bt.feeds.GenericCSVData(
        dataname = 'TSLA.csv',
        dtformat='%Y-%m-%d',
        # setting fot testing data 
        fromdate = datetime.datetime(2024,1,1),
        todate = datetime.datetime(2026,1,1),
        datetime=0,
        open=4,
        high=2,
        low=3,
        close=1,
        volume=5,
        openinterest = -1,
        headers = True
    )
cerebro.adddata(data)
cerebro.addstrategy(AverageTrueRange)

cerebro.run()




c:\Users\vipul\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\backtrader\cerebro.py:675: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
  - callback(msg, \*args, \*\*kwargs)
c:\Users\vipul\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\backtrader\cerebro.py:717: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
  - callback(data, status, \*args, \*\*kwargs)


AttributeError: 'Lines_LineSeries_LineIterator_DataAccessor_StrategyBase_Strategy_AverageTrueRange' object has no attribute 'dataclose'